# **LAB: SPARK**
**`Use Spark to process parquet data`**
- [Spark Syntax & Tuto](https://spark.apache.org/docs/latest/sql-getting-started.html "Spark Syntax & Tuto")
- [Parquet Syntax & Tuto](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html "Parquet Syntax & Tuto")
- [Dataframe](https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html "Dataframe")
- Here we suppose, you already used **`NIFI`** to ingest data
- **Base bucket:** `raw-bucket`
- **Buckets:** `raw-bucket/client`, `raw-bucket/product`, `raw-bucket/sales`.
- **Steps:**
    - Initialize `Spark Session` (`.builder` function)
    - Read parquet files
    - Store in a `dataframe`
    - Create `Temp View`
    - Use `SQL` to query datas

In [13]:
from pyspark.sql import SparkSession

In [14]:
from pyspark.sql import SparkSession

# Configuration alignée sur votre fichier YAML (Master: 7077, MinIO: admin/password123)
# 1 core / executors
# 800 MB RAM / executors
spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("Minio-SQL-Parquet") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.executor.memory", "800m") \
    .config("spark.executor.cores", "1") \
    .config("spark.sql.shuffle.partitions", "2") \
    .config("spark.sql.parquet.filterPushdown", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print("Spark est connecté au Master et à MinIO !")


Spark est connecté au Master et à MinIO !


## Client:

- **bucket name:** `raw-bucket/client`
- Make sure you copy good parquet filename. In my case it's `client_20260311184923.parquet`

### **Init Var**
- **`file_client`:** used to read only one `parquet` file
- **`file_star_client`:** used to read all files under specific folder

In [15]:
file_client = "s3a://raw-bucket/client/client_20260311184923.parquet"
file_star_client = "s3a://raw-bucket/client/*.parquet"

### Read Parquet File
We read parquet file and transform in temp sql view `one_parquet_table_client` and we count number of rows

In [24]:
# 1. Charger les données (assurez-vous d'avoir créé le bucket dans MinIO)
df_one_client = spark.read.parquet(file_client)

In [25]:
# 2. Créer une vue temporaire pour utiliser le SQL standard
df_one_client.createOrReplaceTempView("one_parquet_table_client")

In [27]:
# 3. Exécuter une requête SQL
result_one_client = spark.sql("""
    SELECT count(*) as nombre
    FROM one_parquet_table_client 
""")


In [28]:
result_one_client.show()

+------+
|nombre|
+------+
|    10|
+------+



In [39]:
# Print the schema in a tree format
df_one_client.printSchema()

root
 |-- id: integer (nullable = true)
 |-- code: string (nullable = true)
 |-- name: string (nullable = true)
 |-- actif: boolean (nullable = true)



### Read All client Parquet Files
We read parquet all files and transform in temp sql view `all_client_table` and we process it

In [29]:
# 1. Charger toutes les données (assurez-vous d'avoir créé le bucket dans MinIO)
df_star_client = spark.read.parquet(file_star_client)
# 2. Créer une vue temporaire pour utiliser le SQL standard
df_star_client.createOrReplaceTempView("all_client_table")

In [30]:
# 3. Exécuter une requête SQL
result_star_client = spark.sql("""
    SELECT * 
    FROM all_client_table
    ORDER BY id
""")


In [31]:
result_star_client.show(10)

+---+----+----------------+-----+
| id|code|            name|actif|
+---+----+----------------+-----+
|  1|C001|    Alice Dupont| true|
|  2|C002|   Bertrand SARL| true|
|  3|C003|Clinique du Parc| true|
|  4|C004|     David Morel| true|
|  5|C005|      Ets Durand|false|
|  6|C006|     Fanny Leroy| true|
|  7|C007|  Garage Central| true|
|  8|C008|    Hélène Petit| true|
|  9|C009|       Immo Plus| true|
| 10|C010|    Julien Simon| true|
+---+----+----------------+-----+
only showing top 10 rows



## Sales:

- **bucket name:** `raw-bucket/sales`
- Make sure you copy good parquet filename. In my case it's `sales_20260311191523.parquet`

In [32]:
file_sales = "s3a://raw-bucket/sales/sales_20260311191523.parquet"
file_star_sales = "s3a://raw-bucket/sales/*.parquet"

In [42]:
# 1. Charger les données (assurez-vous d'avoir créé le bucket dans MinIO)
df_sales_one = spark.read.parquet(file_sales)
# 2. Créer une vue temporaire pour utiliser le SQL standard
df_sales_one.createOrReplaceTempView("one_parquet_table_sales")
# 3. Exécuter une requête SQL
result_sales_one = spark.sql("""
    SELECT count(*) as nombre
    FROM one_parquet_table_sales 
""")

In [44]:
result_sales_one.show()

+------+
|nombre|
+------+
|    80|
+------+



In [45]:
# Print the schema in a tree format
df_sales_one.printSchema()

root
 |-- id: integer (nullable = true)
 |-- id_client: integer (nullable = true)
 |-- id_product: integer (nullable = true)
 |-- qte: integer (nullable = true)
 |-- total: string (nullable = true)



### Read All sales Parquet Files
We read parquet all files and transform in temp sql view `all_sales_table` and we process it

In [50]:
# 1. Charger toutes les données (assurez-vous d'avoir créé le bucket dans MinIO)
df_star_sales = spark.read.parquet(file_star_sales)
# 2. Créer une vue temporaire pour utiliser le SQL standard
df_star_sales.createOrReplaceTempView("all_sales_table")

In [51]:
# 3. Exécuter une requête SQL
result_star_sales = spark.sql("""
    SELECT * 
    FROM all_sales_table
    ORDER BY id
    LIMIT 20
""")

In [52]:
result_star_sales.show(10)

+---+---------+----------+---+------+
| id|id_client|id_product|qte| total|
+---+---------+----------+---+------+
|  1|        1|         1|  1|850.00|
|  2|        1|         2|  2| 51.00|
|  3|        2|         3|  1| 75.00|
|  4|        2|         4|  2|300.00|
|  5|        3|         5|  1|210.00|
|  6|        3|         1|  1|850.00|
|  7|        4|         6|  3|135.00|
|  8|        4|         7|  1| 90.00|
|  9|        5|         8| 10|150.00|
| 10|        5|         9|  2|110.00|
+---+---------+----------+---+------+
only showing top 10 rows



## Product (**To Do**):

- **bucket name:** `raw-bucket/product`
- Make sure you copy good parquet filename. In my case it's `!!!.parquet`

## Some Queries & functions:

- [Spark Syntax & Tuto](https://spark.apache.org/docs/latest/sql-getting-started.html "Spark Syntax & Tuto")
- [Parquet Syntax & Tuto](https://spark.apache.org/docs/latest/sql-data-sources-parquet.html "Parquet Syntax & Tuto")

In [48]:
# Select client "code" & "name" columns
df_star_client.select("code", "name").show(2)

+----+---------------+
|code|           name|
+----+---------------+
|C011| Arnaud Bernard|
|C012|Boulangerie Bio|
+----+---------------+
only showing top 2 rows



In [54]:
# Select sales having qte>1
df_star_sales.filter(df_star_sales['qte'] > 2).show(2)

+---+---------+----------+---+------+
| id|id_client|id_product|qte| total|
+---+---------+----------+---+------+
|  7|        4|         6|  3|135.00|
|  9|        5|         8| 10|150.00|
+---+---------+----------+---+------+
only showing top 2 rows



In [55]:
# Count sales by id_client
df_star_sales.groupBy("id_client").count().show(3)

+---------+-----+
|id_client|count|
+---------+-----+
|       18|   11|
|       12|   11|
|       13|   11|
+---------+-----+
only showing top 3 rows



In [58]:
from pyspark.sql import functions as F

# Count sales by id_client and order
df_star_sales.groupBy("id_client").count().orderBy(F.col("count").desc()).show(3)

+---------+-----+
|id_client|count|
+---------+-----+
|       18|   11|
|       12|   11|
|       13|   11|
+---------+-----+
only showing top 3 rows



In [59]:
from pyspark.sql import functions as F

# Best clients in term of bying
df_star_sales.groupBy("id_client") \
  .agg(F.sum("total").alias("somme_totale")) \
  .orderBy(F.col("somme_totale").desc()) \
  .show(3)

+---------+------------+
|id_client|somme_totale|
+---------+------------+
|       20|      1609.0|
|       12|      1410.0|
|       18|      1360.0|
+---------+------------+
only showing top 3 rows



In [62]:
from pyspark.sql import functions as F

# 1. Joindre sur df_star_sales.id_client == df_star_client.id
# 2. Grouper par id_client ET name
# 3. Calculer la somme et trier
r = df_star_sales.join(df_star_client, df_star_sales.id_client == df_star_client.id, "inner") \
    .groupBy("id_client", "name") \
    .agg(F.sum("total").alias("somme_totale")) \
    .orderBy(F.col("somme_totale").desc())

r.show(5)


+---------+---------------+------------+
|id_client|           name|somme_totale|
+---------+---------------+------------+
|       20|   Jean Valjean|      1609.0|
|       12|Boulangerie Bio|      1410.0|
|       18|    Hugo Bossis|      1360.0|
|       17|    Gérard & Co|      1327.0|
|       11| Arnaud Bernard|      1289.0|
+---------+---------------+------------+
only showing top 5 rows



## Some Optimizations (tunning):

- [Spark Syntax & Tuto](https://spark.apache.org/docs/latest/sql-getting-started.html "Spark Syntax & Tuto")
- [Spark tunning](https://spark.apache.org/docs/latest/sql-performance-tuning.html "Spark tunning")

### Partitions
- vérifier le nombre de partitions générées par votre **`groupBy`** ou **`join`** (ou toute opération provoquant un **`shuffle`**),
- **`df.rdd.getNumPartitions()`**

In [67]:
# Count sales by id_client
df_star_sales.groupBy("id_client").count().show(3)
# On vérifie le nombre de partitions créées par le shuffle
print(f"Nombre de partitions : {df_star_sales.rdd.getNumPartitions()}")

+---------+-----+
|id_client|count|
+---------+-----+
|        2|    3|
|        4|    3|
|        5|    3|
+---------+-----+
only showing top 3 rows

Nombre de partitions : 2


### Max Partitions
- **`spark.sql.files.maxPartitionBytes` (128 Mo):**
- SPARK utilise ce paramètre pour décider comment découper les fichiers en "morceaux" (partitions) que les processeurs traiteront en parallèle.
- Un fichier de 1 Go sera découpé en environ 8 partitions de **128 Mo**.
- Augmenter pour de gros fichiers, Reduire pour de petits fichiers
- **Changer la Valeur:** Fixer la taille à 10 Mo `spark.conf.set("spark.sql.files.maxPartitionBytes", 10485760)`
- **Vérifier la valeur actuelle:** `print(spark.conf.get("spark.sql.files.maxPartitionBytes"))`
- **Note importante :** Ce paramètre ne s'applique qu'à la lecture initiale des fichiers.

In [63]:
print(spark.conf.get("spark.sql.files.maxPartitionBytes"))

134217728b


### Shuffle Partitions
- **`spark.sql.shuffle.partitions` (default = 200):**
- nombre de partitions créées lors des opérations de mélange de données (shuffle), comme votre groupBy ou votre join.
- **Valeur par défaut** : `200`
- **Trop élevée:** pour de petits datasets de **l'ordre Ko**, cela crée des centaines de tâches vides ou minuscules, ce qui ralentit tout. Réglez-le à un chiffre bas **`(ex: 4, 8 ou 20)`**.
- **Trop faible:** pour des **téraoctets de données**, les partitions deviennent trop lourdes et causent des erreurs `"Out of Memory" (OOM)`. Réglez-le sur un multiple du nombre total de cœurs (vCores) de vos exécuteurs **`(ex: 2x ou 3x le nombre de cœurs)`**.
- **Changer la Valeur:** Fixer la valeur à 8 `spark.conf.set("spark.sql.shuffle.partitions", "8")`
- **Vérifier la valeur actuelle:** `print(spark.conf.get("spark.sql.shuffle.partitions"))`.

In [68]:
print(spark.conf.get("spark.sql.shuffle.partitions"))

2


### Adaptive Query Execution (AQE)
- Plus besoin de deviner `spark.sql.shuffle.partitions`
- Activez l'`AQE`.

```python
# Active le redimensionnement automatique des partitions de shuffle
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
```

In [71]:
print("Adaptive: "+spark.conf.get("spark.sql.adaptive.enabled"))
print("Adaptive Coalesce: "+spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled"))

Adaptive: true
Adaptive Coalesce: true


In [73]:
spark.stop()

## IA & ML
- **Régression Linéaire**
- **Segmentation K-Means**
- **Remarques:**
    - Lorsque vous avez créé votre DataFrame initial avec **df = spark.read...**, ce DataFrame (df) possède une référence interne vers la SparkSession.
    - Tous les objets qui découlent de ce DataFrame conservent ce lien invisible.
    - **Example `scaler.fit(temp_data)`** : Au moment où on lance cette commande, le code du `StandardScaler` appelle en arrièere plans les API de la `SparkSession` associée à `temp_data` pour distribuer le calcul

### Régression Linéaire
- Prédire le montant du prochain achat (total) ou la quantité (qte) qu'un client va commander.
- Estimer le chiffre d'affaires futur par client.

In [1]:
file_sales = "s3a://raw-bucket/sales/sales_20260311191523.parquet"
file_star_sales = "s3a://raw-bucket/sales/*.parquet"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
#from pyspark.ml.evaluation import RegressionEvaluator

In [3]:
# Configuration alignée sur votre fichier YAML (Master: 7077, MinIO: admin/password123)
# 1 core / executors
# 800 MB RAM / executors
spark_ai = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("AI_Analysis_Client") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.executor.memory", "800m") \
    .config("spark.executor.cores", "1") \
    .config("spark.sql.shuffle.partitions", "2") \
    .config("spark.sql.parquet.filterPushdown", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print("Spark est connecté au Master et à MinIO !")

Spark est connecté au Master et à MinIO !


In [4]:
df_star_sales = spark_ai.read.parquet(file_star_sales)

In [5]:
# --- PRÉPARATION : Agrégation par client ---
# On calcule le total cumulé et la quantité moyenne pour chaque client
client_profile = df_star_sales.groupBy("id_client").agg(
    F.sum("total").alias("total_cumule"),
    F.avg("qte").alias("qte_moyenne"),
    F.count("id_client").alias("frequence")
)

In [6]:
client_profile.printSchema()

root
 |-- id_client: integer (nullable = true)
 |-- total_cumule: double (nullable = true)
 |-- qte_moyenne: double (nullable = true)
 |-- frequence: long (nullable = false)



In [7]:
client_profile.show(3)

+---------+------------+------------------+---------+
|id_client|total_cumule|       qte_moyenne|frequence|
+---------+------------+------------------+---------+
|        2|      1225.0|1.3333333333333333|        3|
|        4|       300.0|1.6666666666666667|        3|
|        5|       410.0| 4.333333333333333|        3|
+---------+------------+------------------+---------+
only showing top 3 rows



In [8]:
# Assemblage des features pour la régression
assembler_reg = VectorAssembler(inputCols=["qte_moyenne", "frequence"], outputCol="features_reg")
data_reg = assembler_reg.transform(client_profile)

# Modèle de régression
lr = LinearRegression(featuresCol="features_reg", labelCol="total_cumule")
lr_model = lr.fit(data_reg)

# Prédiction
predictions_ca = lr_model.transform(data_reg)
predictions_ca.select("id_client", "total_cumule", "prediction").show(5)


+---------+------------+------------------+
|id_client|total_cumule|        prediction|
+---------+------------+------------------+
|        2|      1225.0|  786.991752676965|
|        4|       300.0| 753.5909180688863|
|        5|       410.0|486.38424120425657|
|       10|       980.0| 686.7892488527289|
|       12|      1410.0|1242.3833376758735|
+---------+------------+------------------+
only showing top 5 rows



In [9]:
# Prédiction pour le client ID 5
# On filtre sur l'ID 5 pour voir sa prédiction de quantité ou total
res_client_5 = predictions_ca.filter(F.col("id_client") == 5)
res_client_5.select("id_client", "prediction").show()


+---------+------------------+
|id_client|        prediction|
+---------+------------------+
|        5|486.38424120425657|
+---------+------------------+



In [10]:
# Estimation du chiffre d'affaires futur
res_client_4 = predictions_ca.filter(F.col("id_client") == 4)
ca_estime = res_client_4.select("prediction").collect()[0][0]

print(f"Le CA futur estimé pour le client 4 est de : {ca_estime:.2f} €")

Le CA futur estimé pour le client 4 est de : 753.59 €


### K-MEANS
- Identifier des segments comme les "Clients fidèles à petit budget" ou les "Acheteurs occasionnels de gros volumes".
- **Note:** Le StandardScaler est crucial ici car les échelles de prix et de quantités diffèrent. Si un client dépense 1000 € (total_cumule) et achète 2 produits (qte), le chiffre 1000 va "écraser" le chiffre 2 dans le calcul de distance.

In [11]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler, StandardScaler

In [14]:
# Normalisation des données
assembler_km = VectorAssembler(inputCols=["qte_moyenne", "total_cumule"], outputCol="features_raw")
temp_data = assembler_km.transform(client_profile)

scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled")
scaler_model = scaler.fit(temp_data)
final_km_data = scaler_model.transform(temp_data)

# Création de 3 clusters (ex: Petits, Moyens, Gros acheteurs)
kmeans = KMeans(featuresCol="features_scaled", k=2, seed=1)
km_model = kmeans.fit(final_km_data)

# Affichage des segments
segments = km_model.transform(final_km_data)
segments.select("id_client", "prediction").withColumnRenamed("prediction", "cluster").show(5)


+---------+-------+
|id_client|cluster|
+---------+-------+
|        2|      1|
|        4|      1|
|        5|      0|
|       10|      1|
|       12|      0|
+---------+-------+
only showing top 5 rows



In [16]:
# Identification du segment (Cluster)
res_client_2 = segments.filter(F.col("id_client") == 12)
segment_id = res_client_2.select("prediction").collect()[0][0]

# Mapping logique (exemple)
profils = {0: "Petit budget / Fidèle", 1: "Gros volume / Occasionnel", 2: "Client VIP"}
print(f"Le client 2 appartient au segment {segment_id} ({profils.get(segment_id)})")


Le client 2 appartient au segment 0 (Petit budget / Fidèle)


In [17]:
client_profile.filter(F.col("id_client") == 12).show()

+---------+------------+-----------+---------+
|id_client|total_cumule|qte_moyenne|frequence|
+---------+------------+-----------+---------+
|       12|      1410.0|        3.0|       11|
+---------+------------+-----------+---------+



In [18]:
# Identification du segment (Cluster)
res_client_2 = segments.filter(F.col("id_client") == 5)
segment_id = res_client_2.select("prediction").collect()[0][0]

# Mapping logique (exemple)
profils = {0: "Petit budget / Fidèle", 1: "Gros volume / Occasionnel", 2: "Client VIP"}
print(f"Le client 2 appartient au segment {segment_id} ({profils.get(segment_id)})")

Le client 2 appartient au segment 0 (Petit budget / Fidèle)


In [19]:
client_profile.filter(F.col("id_client") == 5).show()

+---------+------------+-----------------+---------+
|id_client|total_cumule|      qte_moyenne|frequence|
+---------+------------+-----------------+---------+
|        5|       410.0|4.333333333333333|        3|
+---------+------------+-----------------+---------+



In [ ]:
spark_ai.stop()